# 05 – Evaluation: U-Net Crack Detection

**CSE445 – Road Damage Detection & Lane Segmentation**

This notebook:
1. Loads the best U-Net checkpoint from `experiments/crack_unet_run1/`
2. Runs inference on the **held-out test set** (never seen during training)
3. Computes quantitative metrics: IoU, Dice, Pixel Accuracy
4. Visualises predictions vs ground truth
5. Analyses failure cases

**Prerequisites**: Run `03_train_crack_unet.ipynb` first.

In [ ]:
import sys, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/Road_Damage_Project'
    os.environ['RUN_ENV'] = 'colab'
except ImportError:
    REPO = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
    os.environ['RUN_ENV'] = 'local'

if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('Repo root:', REPO)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from tqdm import tqdm

import config
from src.shared.dataset    import SegmentationDataset
from src.shared.transforms import get_val_transforms
from src.shared.unet       import UNet
from src.shared.metrics    import compute_all_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Load Model Checkpoint

In [ ]:
cfg  = config.CRACK_UNET
ckpt = config.EXP_DIR / cfg['run_name'] / 'best_model.pth'
print('Loading checkpoint:', ckpt)

model = UNet(cfg['in_channels'], cfg['out_channels'], cfg['base_features'])
model.load_state_dict(torch.load(ckpt, map_location=device))
model = model.to(device)
model.eval()
print('Model loaded. Parameters:', f"{model.count_parameters():,}")

## 2. Test DataLoader

In [ ]:
test_ds = SegmentationDataset(
    config.CRACK_SPLIT_DIR / 'test.csv',
    transform=get_val_transforms((config.IMG_HEIGHT, config.IMG_WIDTH))
)
test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=2)
print(f'Test set: {len(test_ds)} samples, {len(test_loader)} batches')

## 3. Quantitative Evaluation on Test Set

In [ ]:
all_iou, all_dice, all_acc = [], [], []

with torch.no_grad():
    for images, masks in tqdm(test_loader, desc='Evaluating'):
        images = images.to(device)
        masks  = masks.to(device)
        preds  = model(images)
        m      = compute_all_metrics(preds, masks)
        all_iou.append(m['iou'])
        all_dice.append(m['dice'])
        all_acc.append(m['pixel_acc'])

mean_iou  = np.mean(all_iou)
mean_dice = np.mean(all_dice)
mean_acc  = np.mean(all_acc)

print('\n' + '='*50)
print('  CRACK DETECTION – TEST SET RESULTS')
print('='*50)
print(f'  IoU (Jaccard)  : {mean_iou:.4f}')
print(f'  Dice (F1)      : {mean_dice:.4f}')
print(f'  Pixel Accuracy : {mean_acc:.4f}')
print('='*50)

# Bar chart
fig, ax = plt.subplots(figsize=(7, 4))
metrics = ['IoU', 'Dice', 'Pixel Acc']
values  = [mean_iou, mean_dice, mean_acc]
colors  = ['#1565C0', '#2E7D32', '#6A1B9A']
bars = ax.bar(metrics, values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.set_title('U-Net Crack Detection – Test Set Metrics')
plt.tight_layout(); plt.show()

## 4. Visual Predictions vs Ground Truth

In [ ]:
mean_arr = np.array([0.485, 0.456, 0.406])
std_arr  = np.array([0.229, 0.224, 0.225])

N = 8
fig, axes = plt.subplots(N, 4, figsize=(14, N * 3.2))
fig.suptitle('U-Net Crack Detection – Predictions vs Ground Truth', fontsize=13, fontweight='bold')

batch_imgs, batch_masks = next(iter(DataLoader(test_ds, batch_size=N, shuffle=True)))
with torch.no_grad():
    batch_preds = model(batch_imgs.to(device)).cpu()

for i in range(N):
    img_np    = (batch_imgs[i].permute(1,2,0).numpy() * std_arr + mean_arr).clip(0, 1)
    mask_np   = batch_masks[i].squeeze().numpy()
    pred_np   = (batch_preds[i].squeeze().numpy() > 0.5).astype(np.float32)

    # Difference map
    diff       = np.zeros((*pred_np.shape, 3))
    diff[..., 0] = np.maximum(pred_np - mask_np, 0)   # false positive → red
    diff[..., 2] = np.maximum(mask_np - pred_np, 0)   # false negative → blue

    axes[i, 0].imshow(img_np);              axes[i, 0].set_title('Image')
    axes[i, 1].imshow(mask_np, cmap='gray'); axes[i, 1].set_title('Ground Truth')
    axes[i, 2].imshow(pred_np, cmap='gray'); axes[i, 2].set_title('Prediction')
    axes[i, 3].imshow(diff);               axes[i, 3].set_title('Error (R=FP, B=FN)')
    for ax in axes[i]: ax.axis('off')

plt.tight_layout(); plt.show()

## 5. Failure Case Analysis

In [ ]:
# Collect per-sample IoU across the test set
from src.shared.metrics import iou_score

sample_ious = []
with torch.no_grad():
    for images, masks in test_loader:
        preds = model(images.to(device)).cpu()
        for i in range(preds.shape[0]):
            sample_iou = iou_score(preds[i:i+1], masks[i:i+1])
            sample_ious.append(sample_iou)

sample_ious = np.array(sample_ious)
worst_idx   = np.argsort(sample_ious)[:6]   # 6 worst predictions

print(f'Worst 6 sample IoUs: {sample_ious[worst_idx]}')

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('Failure Cases – Lowest IoU Predictions', fontsize=12)

for col, idx in enumerate(worst_idx):
    img_t, mask_t = test_ds[idx]
    with torch.no_grad():
        pred_t = model(img_t.unsqueeze(0).to(device)).squeeze().cpu()

    img_np  = (img_t.permute(1,2,0).numpy() * std_arr + mean_arr).clip(0, 1)
    pred_np = (pred_t.numpy() > 0.5).astype(np.float32)
    mask_np = mask_t.squeeze().numpy()

    axes[0, col].imshow(img_np);               axes[0, col].set_title(f'IoU={sample_ious[idx]:.3f}')
    axes[1, col].imshow(np.stack([pred_np, mask_np, np.zeros_like(pred_np)], axis=-1))
    axes[1, col].set_title('Green=GT  Red=Pred')
    for row in range(2): axes[row, col].axis('off')

plt.tight_layout(); plt.show()

## Summary

Record your test set results here:

| Metric | Score |
|---|---|
| IoU (Jaccard) | __ |
| Dice (F1) | __ |
| Pixel Accuracy | __ |

**Next**: `06_evaluate_lane.ipynb`